In [ ]:
#Stage 2

In [10]:
import os
import re
import argparse
import pandas as pd
from rdflib import Graph


In [11]:
CAPACITY_QUERY = """
PREFIX brick: <https://brickschema.org/schema/Brick#>
PREFIX ext: <https://www.HKUST_PV_System.com/schema/BrickExtension#>
PREFIX unit: <http://qudt.org/vocab/unit/>
 
SELECT ?site ?ratedPowerKW
WHERE {
    ?site a brick:PV_Generation_System ;
          ext:ratedPowerOutput ?powerNode .
    ?powerNode brick:hasUnit unit:KW ;
               brick:value ?ratedPowerKW .
}
"""

In [12]:
ttl_path = ("data/Dataset/Metadata/PV generation system metadata.ttl")

g = Graph()
g.parse(ttl_path, format="turtle")


<Graph identifier=N58becdb58e524d86b6b81ad72b1b780c (<class 'rdflib.graph.Graph'>)>

In [13]:
rows = []
for row in g.query(CAPACITY_QUERY):
        site_uri = str(row.site)
        site_id = site_uri.split("#")[-1]  # local name after the # in the URI
        rated_kw = float(row.ratedPowerKW)
        # print(f"Site id: {site_id} = rated power: {rated_kw}")

        rows.append((site_id, rated_kw))

capacity_df = pd.DataFrame(rows, columns=["site_id_ttl", "rated_power_kw"])

print(capacity_df)


                       site_id_ttl  rated_power_kw
0                  CYT__Station_1_           25.00
1                  CYT__Station_2_           25.00
2              Indoor_Sport_Center           33.30
3                        LSK_North           55.00
4                        LSK_South           25.00
5                          Library          170.20
6                              SQ1           27.60
7                             SQ10           30.40
8                             SQ11           32.51
9                             SQ12           32.12
10                            SQ13           28.47
11                            SQ14           32.90
12                            SQ15           33.29
13                            SQ16           33.15
14                            SQ17           37.05
15                            SQ18           33.93
16                            SQ19           32.51
17                             SQ2           33.30
18                             

In [16]:
site_mapping = {
    'S H Ho Sports Hall': 'S_H_Ho_Sports_Hall',
    'SQ567': 'SQ567',
    'SQ Block P': 'SQ_Block_P',
    'UG Hall3': 'UG_Hall3',
    'UG Hall9': 'UG_Hall9',
    'Zone A5': 'Zone_A5',
    'Zone J2': 'Zone_J2',
    'Library': 'Library',
    'SQ1': 'SQ1',
    'SQ Apartment1-12': 'SQ_Apartment1_12',
    'SQ Block R': 'SQ_Block_R',
    'UG Hall4': 'UG_Hall4',
    'Wong Check She Research Centre': 'Wong_Check_She_Research_Centre',
    'Zone A6': 'Zone_A6',
    'Zone L2': 'Zone_L2',
    'LSK North': 'LSK_North',
    'SQ2': 'SQ2',
    'SQ Apartment13-24': 'SQ_Apartment13_24',
    'SQ Block S': 'SQ_Block_S',
    'UG Hall6': 'UG_Hall6',
    'Zone A1': 'Zone_A1',
    'Zone A7': 'Zone_A7',
    'LSK South': 'LSK_South',
    'SQ3': 'SQ3',
    'SQ Apartment25-36': 'SQ_Apartment25_36',
    'UG Hall2 2F': 'UG_Hall2_2F',
    'UG Hall7': 'UG_Hall7',
    'Zone A2': 'Zone_A2',
    'Zone D': 'Zone_D',
    'Shaw Auditorium': 'Shaw_Auditorium',
    'SQ4': 'SQ4',
    'SQ Apartment37-48': 'SQ_Apartment37_48',
    'UG Hall2 RF': 'UG_Hall2_RF',
    'UG Hall8': 'UG_Hall8',
    'Zone A4': 'Zone_A4',
    'Zone J1': 'Zone_J1',
    'Indoor Sports Centre': 'Indoor_Sport_Center',   # manual fix: "Sports" vs "Sport"
}

In [14]:
pv_gen_df = pd.read_csv("data/stage-1/merged_solar_data.csv")

print(pv_gen_df.columns)

Index(['Time', 'generation(kWh)', 'power(W)', 'site'], dtype='object')


In [17]:
# --- map generation 'site' (filename-based) to site_id_ttl ---
pv_gen_df["site_id_ttl"] = pv_gen_df["site"].map(site_mapping)

# --- merge in rated_power_kw ---
pv_gen_df = pv_gen_df.merge(
    capacity_df[["site_id_ttl", "rated_power_kw"]],
    on="site_id_ttl",
    how="left"
)

# --- sanity checks ---
unmapped = pv_gen_df[pv_gen_df["site_id_ttl"].isna()]["site"].unique()
no_capacity = pv_gen_df[pv_gen_df["rated_power_kw"].isna()]["site"].unique()

print("Sites with no mapping entry:", unmapped)
print("Sites mapped but missing in capacity_df:", no_capacity)

pv_gen_df.head()

Sites with no mapping entry: []
Sites mapped but missing in capacity_df: []


,Time,generation(kWh),power(W),site,site_id_ttl,rated_power_kw
0,2021-08-01 00:00:00,0.0,0.0,SQ Apartment25-36,SQ_Apartment25_36,17.0
1,2021-08-01 00:15:00,0.0,0.0,SQ Apartment25-36,SQ_Apartment25_36,17.0
2,2021-08-01 00:30:00,0.0,0.0,SQ Apartment25-36,SQ_Apartment25_36,17.0
3,2021-08-01 00:45:00,0.0,0.0,SQ Apartment25-36,SQ_Apartment25_36,17.0
4,2021-08-01 01:00:00,0.0,0.0,SQ Apartment25-36,SQ_Apartment25_36,17.0


In [18]:
pv_gen_df.describe(include='all')

,Time,generation(kWh),power(W),site,site_id_ttl,rated_power_kw
count,2779008,2.761001e+06,2.761020e+06,2779008,2779008,2.779008e+06
unique,178368,NaN,NaN,37,37,NaN
top,2023-08-01 03:45:00,NaN,NaN,LSK North,LSK_North,NaN
freq,34,NaN,NaN,90624,90624,NaN
mean,NaN,1.263326e+00,4.909581e+03,NaN,NaN,4.275872e+01
std,NaN,2.864391e+00,1.117303e+04,NaN,NaN,4.556270e+01
min,NaN,0.000000e+00,0.000000e+00,NaN,NaN,8.000000e+00
25%,NaN,0.000000e+00,0.000000e+00,NaN,NaN,2.340000e+01
50%,NaN,0.000000e+00,0.000000e+00,NaN,NaN,3.285000e+01
75%,NaN,1.302000e+00,5.020000e+03,NaN,NaN,4.797000e+01


In [19]:
# Load weather data
weather_df = pd.read_csv("data/stage-1/merged_meteorological_data.csv")

In [20]:
# Try to parse Time column strictly as YYYY-MM-DD HH:MM:SS
pv_gen_df['Time_parsed'] = pd.to_datetime(pv_gen_df['Time'], format='%Y-%m-%d %H:%M:%S', errors='coerce')

# Keep only rows where parsing succeeded (i.e., matched the exact format)
pv_gen_df_clean = pv_gen_df[pv_gen_df['Time_parsed'].notna()].copy()

# Also validate generation/power are numeric
pv_gen_df_clean['generation(kWh)'] = pd.to_numeric(pv_gen_df_clean['generation(kWh)'], errors='coerce')
pv_gen_df_clean['power(W)'] = pd.to_numeric(pv_gen_df_clean['power(W)'], errors='coerce')
pv_gen_df_clean = pv_gen_df_clean.dropna(subset=['generation(kWh)', 'power(W)'])

# Drop helper column, keep original Time string (or replace with parsed one)
pv_gen_df_clean = pv_gen_df_clean.drop(columns=['Time_parsed'])

pv_gen_df_clean.to_csv('cleaned_output.csv', index=False)
print(f"Kept {len(pv_gen_df_clean)} of {len(pv_gen_df)} rows")

Kept 2556901 of 2779008 rows


In [21]:
# Ensure datetime types
pv_gen_df_clean['Time'] = pd.to_datetime(pv_gen_df_clean['Time'])
weather_df['datetime'] = pd.to_datetime(weather_df['datetime'])

# Merge on matching timestamps
merged_df = pd.merge(
    pv_gen_df_clean,
    weather_df,
    left_on='Time',
    right_on='datetime',
    how='inner'  # use 'left' to keep all PV rows even if weather is missing
)

print(merged_df.shape)
merged_df.head()

(2536471, 15)


,Time,generation(kWh),power(W),site,site_id_ttl,rated_power_kw,datetime,Irradiance_Irradiance (W/m2),Rainfall_Rainfall(mm),Relative_Humidity_RH (%),Sea_Level_Pressure_SLP (hPa),Temperature_Temp (Degree Celsius),Visibility_Vis (km),Wind_Wind Speed (m/s),Wind_Wind Direction (degree)
0,2021-08-01 00:00:00,0.0,0.0,SQ Apartment25-36,SQ_Apartment25_36,17.0,2021-08-01 00:00:00,6.0610,0.254,88.703,1001.79913,28.253,8.8796,1.81470,358.56
1,2021-08-01 00:15:00,0.0,0.0,SQ Apartment25-36,SQ_Apartment25_36,17.0,2021-08-01 00:15:00,5.2896,0.000,91.764,1001.77913,27.426,15.9950,3.66970,337.83
2,2021-08-01 00:30:00,0.0,0.0,SQ Apartment25-36,SQ_Apartment25_36,17.0,2021-08-01 00:30:00,4.8487,0.000,93.888,1001.89913,26.242,15.9950,0.87607,310.66
3,2021-08-01 00:45:00,0.0,0.0,SQ Apartment25-36,SQ_Apartment25_36,17.0,2021-08-01 00:45:00,4.9591,0.000,95.816,1001.81913,26.850,14.0990,2.19480,173.88
4,2021-08-01 01:00:00,0.0,0.0,SQ Apartment25-36,SQ_Apartment25_36,17.0,2021-08-01 01:00:00,5.5102,0.000,96.053,1001.76913,26.605,15.9940,1.30530,215.44


In [ ]:
merged_df['datetime'] = pd.to_datetime(merged_df['datetime'])

# --- Method 1: Normalize generation (kWh) by rated capacity ---
# This gives a "capacity factor" style normalized generation
# Useful when you have generation over an interval and want it independent of plant size
merged_df['normalized_generation'] = merged_df['generation(kWh)'] / merged_df['rated_power_kw']

# --- Method 2: Normalize power (W) by rated power ---
# Convert rated_power_kw to W for consistent units, then normalize
merged_df['rated_power_w'] = merged_df['rated_power_kw'] * 1000
merged_df['normalized_power'] = merged_df['power(W)'] / merged_df['rated_power_w']

# Optional: clip values to [0, 1] in case of sensor noise/overproduction spikes
merged_df['normalized_generation'] = merged_df['normalized_generation'].clip(lower=0, upper=1)
merged_df['normalized_power'] = merged_df['normalized_power'].clip(lower=0, upper=1)


,Time,site,generation(kWh),rated_power_kw,normalized_generation,power(W),normalized_power
0,2021-08-01 00:00:00,SQ Apartment25-36,0.0,17.0,0.0,0.0,0.0
1,2021-08-01 00:15:00,SQ Apartment25-36,0.0,17.0,0.0,0.0,0.0
2,2021-08-01 00:30:00,SQ Apartment25-36,0.0,17.0,0.0,0.0,0.0
3,2021-08-01 00:45:00,SQ Apartment25-36,0.0,17.0,0.0,0.0,0.0
4,2021-08-01 01:00:00,SQ Apartment25-36,0.0,17.0,0.0,0.0,0.0


In [29]:
merged_df[['Time', 'site', 'generation(kWh)', 'rated_power_kw', 
           'normalized_generation', 'power(W)', 'normalized_power']].head(50)

,Time,site,generation(kWh),rated_power_kw,normalized_generation,power(W),normalized_power
0,2021-08-01 00:00:00,SQ Apartment25-36,0.000,17.0,0.000000,0.000000,0.000000
1,2021-08-01 00:15:00,SQ Apartment25-36,0.000,17.0,0.000000,0.000000,0.000000
2,2021-08-01 00:30:00,SQ Apartment25-36,0.000,17.0,0.000000,0.000000,0.000000
3,2021-08-01 00:45:00,SQ Apartment25-36,0.000,17.0,0.000000,0.000000,0.000000
4,2021-08-01 01:00:00,SQ Apartment25-36,0.000,17.0,0.000000,0.000000,0.000000
5,2021-08-01 01:15:00,SQ Apartment25-36,0.000,17.0,0.000000,0.000000,0.000000
6,2021-08-01 01:30:00,SQ Apartment25-36,0.000,17.0,0.000000,0.000000,0.000000
7,2021-08-01 01:45:00,SQ Apartment25-36,0.000,17.0,0.000000,0.000000,0.000000
8,2021-08-01 02:00:00,SQ Apartment25-36,0.000,17.0,0.000000,0.000000,0.000000
9,2021-08-01 02:15:00,SQ Apartment25-36,0.000,17.0,0.000000,0.000000,0.000000


In [30]:
merged_df.to_csv('data/stage-2/weather_pv_merged_and_normalised.csv', index=False, )

In [32]:
merged_slim_df = merged_df.loc[:, ('Time', 'Irradiance_Irradiance (W/m2)', 'Rainfall_Rainfall(mm)', 'Relative_Humidity_RH (%)',
'Sea_Level_Pressure_SLP (hPa)', 'Temperature_Temp (Degree Celsius)', 'Visibility_Vis (km)', 'Wind_Wind Speed (m/s)', 'normalized_generation')]

merged_slim_df.to_csv('data/stage-2/slim_weather_pv_merged_and_normalised.csv', index=False, )